In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ══════════════════════════════════════════════════════════════
# STEP 1 — LOAD RAW DATA
# ══════════════════════════════════════════════════════════════
df_raw = pd.read_excel(r"C:\Users\ODAMA\Downloads\07_MTN_Messy_Data_Cleaning.xlsx",
                       sheet_name='Raw Customer Data',
                       header=3)

print("BEFORE CLEANING:")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
print(f"Missing values:\n{df_raw.isnull().sum()}")
print(f"Duplicates: {df_raw.duplicated().sum()}")

# ══════════════════════════════════════════════════════════════
# STEP 2 — MAKE A COPY TO CLEAN
# ══════════════════════════════════════════════════════════════
df = df_raw.copy()

# ══════════════════════════════════════════════════════════════
# STEP 3 — FIX DUPLICATES
# ══════════════════════════════════════════════════════════════
before_rows = len(df)
df = df.drop_duplicates(subset='subscriber_id', keep='first')
print(f"\nDuplicates removed: {before_rows - len(df)}")

# ══════════════════════════════════════════════════════════════
# STEP 4 — FIX DATE FORMAT
# ══════════════════════════════════════════════════════════════
df['date_joined'] = pd.to_datetime(df['date_joined'], errors='coerce', dayfirst=True)
print(f"Invalid dates set to NaT: {df['date_joined'].isnull().sum()}")

# ══════════════════════════════════════════════════════════════
# STEP 5 — FIX FULL NAME
# ══════════════════════════════════════════════════════════════
# Replace bad values with NaN then fill with Unknown
df['full_name'] = df['full_name'].replace(['N/A','UNKNOWN','?????',''], pd.NA)
df['full_name'] = df['full_name'].str.strip().str.title()
df['full_name'] = df['full_name'].fillna('Unknown')

# ══════════════════════════════════════════════════════════════
# STEP 6 — FIX STATE (inconsistent casing + bad values)
# ══════════════════════════════════════════════════════════════
df['state'] = df['state'].replace(['N/A','Unknown','Lago','Rivers State',''], pd.NA)
df['state'] = df['state'].str.strip().str.title()
df['state'] = df['state'].fillna('Unknown')

# ══════════════════════════════════════════════════════════════
# STEP 7 — FIX PLAN TYPE (inconsistent casing + bad values)
# ══════════════════════════════════════════════════════════════
# Standardize all plan names
plan_map = {
    'mtn pulse':     'MTN Pulse',
    'MTN PULSE':     'MTN Pulse',
    'Pulse':         'MTN Pulse',
    'xtratime':      'MTN XtraTime',
    'MTN XtraTime':  'MTN XtraTime',
    'MTN XtraTalk':  'MTN XtraTalk',
    'MTN Business':  'MTN Business',
    'Unknown':        pd.NA,
    'N/A':            pd.NA,
    '':               pd.NA,
}
df['plan_type'] = df['plan_type'].replace(plan_map)
df['plan_type'] = df['plan_type'].fillna('Unknown')

# ══════════════════════════════════════════════════════════════
# STEP 8 — FIX NEGATIVE & OUTLIER VALUES
# ══════════════════════════════════════════════════════════════
# sms_count — negative values not valid
df.loc[df['sms_count'] < 0, 'sms_count'] = pd.NA

# call_minutes — outlier (99999 is clearly wrong)
df.loc[df['call_minutes'] > 10000, 'call_minutes'] = pd.NA

# data_usage_gb — outlier (9999 is clearly wrong)
df.loc[df['data_usage_gb'] > 500, 'data_usage_gb'] = pd.NA

# monthly_recharge — negative values not valid
df.loc[df['monthly_recharge (₦)'] < 0, 'monthly_recharge (₦)'] = pd.NA

# ══════════════════════════════════════════════════════════════
# STEP 9 — FIX STATUS (inconsistent casing)
# ══════════════════════════════════════════════════════════════
status_map = {
    'active':     'Active',
    'ACTIVE':     'Active',
    'Active':     'Active',
    'inactive':   'Inactive',
    'Inactive':   'Inactive',
    'suspended':  'Suspended',
    'Suspended':  'Suspended',
    'Ported Out': 'Ported Out',
    'N/A':         pd.NA,
    '':            pd.NA,
}
df['status'] = df['status'].replace(status_map)
df['status'] = df['status'].fillna('Unknown')

# ══════════════════════════════════════════════════════════════
# STEP 10 — FILL REMAINING MISSING VALUES
# ══════════════════════════════════════════════════════════════
df['monthly_recharge (₦)'] = df['monthly_recharge (₦)'].fillna(df['monthly_recharge (₦)'].median())
df['data_usage_gb']         = df['data_usage_gb'].fillna(df['data_usage_gb'].median())
df['call_minutes']          = df['call_minutes'].fillna(df['call_minutes'].median())
df['sms_count']             = df['sms_count'].fillna(df['sms_count'].median())

# ══════════════════════════════════════════════════════════════
# STEP 11 — SAVE CLEANED DATA
# ══════════════════════════════════════════════════════════════
df.to_excel('MTN_Cleaned_Data.xlsx', index=False)
print(f"\nAFTER CLEANING:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Missing values:\n{df.isnull().sum()}")
print("\n✅ Cleaned data saved as MTN_Cleaned_Data.xlsx")

# ══════════════════════════════════════════════════════════════
# STEP 12 — CHART DATA
# ══════════════════════════════════════════════════════════════

# 1. Status distribution
status_count = df.groupby('status')['subscriber_id'].count().reset_index()
status_count.columns = ['Status', 'Count']

# 2. Plan type distribution
plan_count = df.groupby('plan_type')['subscriber_id'].count().reset_index()
plan_count.columns = ['Plan Type', 'Count']

# 3. Subscribers by State
state_count = df.groupby('state')['subscriber_id'].count().reset_index()
state_count.columns = ['State', 'Count']
state_count = state_count[state_count['State'] != 'Unknown'].sort_values('Count')

# 4. Avg Monthly Recharge by Plan
avg_recharge = df.groupby('plan_type')['monthly_recharge (₦)'].mean().reset_index()
avg_recharge.columns = ['Plan Type', 'Avg Recharge (₦)']
avg_recharge = avg_recharge.sort_values('Avg Recharge (₦)')

# 5. Data Usage vs Call Minutes scatter
# used directly from df

# 6. Missing values before vs after
issues_before = {
    'Duplicates':      df_raw.duplicated().sum(),
    'Missing Dates':   df_raw['date_joined'].isnull().sum(),
    'Missing Names':   df_raw['full_name'].isnull().sum(),
    'Bad Status':      df_raw['status'].isin(['active','ACTIVE','inactive','suspended']).sum(),
    'Negative Values': (df_raw['sms_count'] < 0).sum(),
    'Outliers':        (df_raw['data_usage_gb'] > 500).sum() + (df_raw['call_minutes'] > 10000).sum(),
}
issues_df = pd.DataFrame({
    'Issue': list(issues_before.keys()),
    'Count': list(issues_before.values())
})

# ══════════════════════════════════════════════════════════════
# STEP 13 — CHARTS
# ══════════════════════════════════════════════════════════════

def style(fig):
    fig.update_layout(
        plot_bgcolor='#ffffff',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial', size=13),
        title_font=dict(size=16, color='#1f3b5c', family='Arial'),
        margin=dict(l=20, r=20, t=50, b=100),
        height=400,
        showlegend=True,
        legend=dict(
            bgcolor='rgba(255,255,255,0.95)',
            bordercolor='#d6e6f2',
            borderwidth=1,
            font=dict(size=12, color='#1f3b5c'),
            orientation='h',
            yanchor='top',
            y=-0.25,
            xanchor='center',
            x=0.5
        ),
        xaxis=dict(tickangle=45, tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8'),
        yaxis=dict(tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8')
    )
    return fig

# 1. Issues found bar chart
fig_issues = px.bar(issues_df, x='Issue', y='Count',
                    color='Count', color_continuous_scale='Reds',
                    title='Data Issues Found Before Cleaning')
style(fig_issues)

# 2. Status distribution
fig_status = px.pie(status_count, names='Status', values='Count',
                    hole=0.5, title='Subscriber Status (After Cleaning)',
                    color='Status',
                    color_discrete_map={
                        'Active':     '#1A7A4A',
                        'Inactive':   '#E67E22',
                        'Suspended':  '#C0392B',
                        'Ported Out': '#7F8C8D',
                        'Unknown':    '#BDC3C7'
                    })
style(fig_status)

# 3. Plan type distribution
fig_plan = px.bar(plan_count, x='Plan Type', y='Count',
                  color='Plan Type',
                  color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef','#deebf7'],
                  title='Subscribers by Plan Type')
style(fig_plan)

# 4. Subscribers by State
fig_state = px.bar(state_count, x='Count', y='State',
                   orientation='h', color='Count',
                   color_continuous_scale='Blues',
                   title='Subscribers by State')
style(fig_state)

# 5. Avg Recharge by Plan
fig_recharge = px.bar(avg_recharge, x='Avg Recharge (₦)', y='Plan Type',
                      orientation='h', color='Avg Recharge (₦)',
                      color_continuous_scale='Blues',
                      title='Avg Monthly Recharge by Plan Type')
style(fig_recharge)

# 6. Scatter — Data Usage vs Call Minutes
fig_scatter = px.scatter(df, x='call_minutes', y='data_usage_gb',
                         color='status',
                         size='monthly_recharge (₦)',
                         hover_data=['full_name','plan_type','state'],
                         title='Data Usage vs Call Minutes',
                         color_discrete_map={
                             'Active':     '#1A7A4A',
                             'Inactive':   '#E67E22',
                             'Suspended':  '#C0392B',
                             'Ported Out': '#7F8C8D',
                             'Unknown':    '#BDC3C7'
                         })
fig_scatter.update_traces(marker=dict(
    line=dict(width=2, color='black'),
    opacity=0.85
))
style(fig_scatter)

# ══════════════════════════════════════════════════════════════
# STEP 14 — DASHBOARD
# ══════════════════════════════════════════════════════════════
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE,
    'padding': '16px 22px',
    'borderRadius': '12px',
    'textAlign': 'center',
    'flex': '1',
    'margin': '6px',
    'border': f'1.5px solid {BLUE_LIGHT}',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE,
    'padding': '10px',
    'borderRadius': '12px',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
    'flex': '1',
    'border': f'1px solid {BLUE_LIGHT}'
}
sidebar_label = {
    'backgroundColor': BLUE_MID,
    'color': WHITE,
    'padding': '7px 12px',
    'borderRadius': '6px',
    'marginBottom': '7px',
    'fontSize': '13px',
    'fontWeight': 'bold',
    'textAlign': 'center'
}

app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H2('MTN NIGERIA — DATA CLEANING DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '20px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '16px 22px',
                       'borderRadius': '12px', 'flex': '2', 'marginRight': '12px'}
            ),
            html.Div([
                html.Div([
                    html.P('TOTAL SUBSCRIBERS', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{len(df):,}', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('DUPLICATES REMOVED', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{before_rows - len(df):,}', style={'margin':'0','color':'#C0392B','fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('ISSUES FIXED', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{sum(issues_before.values()):,}', style={'margin':'0','color':'#E67E22','fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('ACTIVE SUBSCRIBERS', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{len(df[df["status"]=="Active"]):,}', style={'margin':'0','color':'#1A7A4A','fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '16px'}),

        # Body
        html.Div([

            # Sidebar
            html.Div([
                html.P('Status', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                html.Div('Active',    style={**sidebar_label, 'backgroundColor': '#1A7A4A'}),
                html.Div('Inactive',  style={**sidebar_label, 'backgroundColor': '#E67E22'}),
                html.Div('Suspended', style={**sidebar_label, 'backgroundColor': '#C0392B'}),
                html.Br(),
                html.P('Plans', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(p, style={**sidebar_label, 'fontSize': '11px'}) for p in ['MTN Pulse','MTN XtraTime','MTN XtraTalk','MTN Business']],
                html.Br(), html.Br(),
                html.P('Prepared by', style={'fontSize':'12px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'13px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '160px', 'minWidth': '160px',
                'backgroundColor': WHITE, 'padding': '16px',
                'borderRadius': '12px', 'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
                'marginRight': '12px', 'border': f'1px solid {BLUE_LIGHT}'
            }),

            # Charts — 2 per row
            html.Div([
                html.Div([
                    html.Div(dcc.Graph(figure=fig_issues,   config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_status,   config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_plan,     config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_state,    config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_recharge, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_scatter,  config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px'}),

            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

if __name__ == '__main__':
    app.run(debug=True, port=8057)

BEFORE CLEANING:
Rows: 200, Columns: 10
Missing values:
subscriber_id            0
date_joined             51
full_name                7
state                   23
plan_type               48
monthly_recharge (₦)     9
data_usage_gb            4
call_minutes             8
sms_count               10
status                  43
dtype: int64
Duplicates: 0

Duplicates removed: 17
Invalid dates set to NaT: 138

AFTER CLEANING:
Rows: 183, Columns: 10
Missing values:
subscriber_id             0
date_joined             138
full_name                 0
state                     0
plan_type                 0
monthly_recharge (₦)      0
data_usage_gb             0
call_minutes              0
sms_count                 0
status                    0
dtype: int64

✅ Cleaned data saved as MTN_Cleaned_Data.xlsx
